# 03 Calibration And CI

Generated by `scripts/revision/create_revision_notebooks.py`.


## Setup

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import zipfile

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = DRIVE_ROOT / 'ECG-RAMBA'
LOCAL_RUNTIME_ROOT = Path('/content/ecg_ramba_runtime')

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
chapman_candidates = [
    DRIVE_ROOT / 'WFDB-ChapmanShaoxing.zip',
    DRIVE_ROOT / 'WFDB_ChapmanShaoxing.zip',
    DRIVE_ROOT / 'chapman.zip',
    DRIVE_ROOT / 'archive.zip',
]
chapman_zip = next((path for path in chapman_candidates if path.is_file()), None)
if chapman_zip is None:
    visible_zips = sorted(path.name for path in DRIVE_ROOT.glob('*.zip')) if DRIVE_ROOT.exists() else []
    raise FileNotFoundError(
        'Chapman dataset ZIP was not found. Checked: '
        + ', '.join(str(path) for path in chapman_candidates)
        + f'. ZIP files currently visible under {DRIVE_ROOT}: {visible_zips}'
    )
os.environ['ECG_RAMBA_CHAPMAN_ZIP'] = str(chapman_zip)
if chapman_zip.stat().st_size == 0 or not zipfile.is_zipfile(chapman_zip):
    raise ValueError(f'Chapman dataset is not a valid non-empty ZIP: {chapman_zip}')
model_dir = DRIVE_ROOT / 'model'
if not model_dir.is_dir():
    raise FileNotFoundError(f'Model directory not found: {model_dir}')
os.environ['ECG_RAMBA_MODEL_DIR'] = str(model_dir)
os.environ['ECG_RAMBA_LOCAL_ROOT'] = str(LOCAL_RUNTIME_ROOT)
os.environ['ECG_RAMBA_EXTRACT_DIR'] = str(LOCAL_RUNTIME_ROOT / 'chapman')
os.environ['ECG_RAMBA_USE_CLEAN_CACHE'] = '0'
os.environ['ECG_RAMBA_SAVE_CLEAN_CACHE'] = '0'

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}', flush=True)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def _git_setup(cmd, check=True):
    return _run_setup(cmd, cwd=REPO_DIR, check=check)

def _git_current_commit():
    result = _git_setup('git rev-parse --short HEAD', check=False)
    return result.stdout.strip() if result.returncode == 0 and result.stdout else 'unknown'

def _pull_or_continue(branch):
    before = _git_current_commit()
    status_result = _git_setup('git status --porcelain', check=False)
    status = status_result.stdout.strip() if status_result.stdout else ''
    if status:
        print('Local repo has changes before pull:')
        print(status[:4000])
    result = _git_setup(f'git pull --ff-only --autostash origin {branch}', check=False)
    after = _git_current_commit()
    if result.returncode:
        print('')
        print('=' * 80)
        print('WARNING: git pull failed; continuing with the currently checked-out repo.')
        print('This avoids deleting Drive files while long-running artifacts may exist.')
        print(f'Current commit: {after} (before pull: {before})')
        print('To force a clean code checkout later, rename the Drive repo folder or use a fresh clone.')
        print('=' * 80)
    else:
        print(f'Git update OK: {before} -> {after}')

if (REPO_DIR / '.git').exists():
    os.chdir(REPO_DIR)
    fetch_result = _git_setup('git fetch origin', check=False)
    if fetch_result.returncode:
        print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
    current_branch_result = _git_setup('git branch --show-current', check=False)
    current_branch = current_branch_result.stdout.strip() if current_branch_result.stdout else ''
    if current_branch != BRANCH:
        checkout_result = _git_setup(f'git checkout {BRANCH}', check=False)
        if checkout_result.returncode:
            print(f'WARNING: git checkout {BRANCH} failed; continuing on branch {current_branch or "<detached>"}')
        else:
            current_branch = BRANCH
    if fetch_result.returncode == 0:
        _pull_or_continue(BRANCH)
elif (REPO_DIR / 'configs' / 'config.py').exists():
    os.chdir(REPO_DIR)
    print('Repo directory exists but is not a git checkout; skipping git pull.')
elif Path.cwd().joinpath('configs', 'config.py').exists():
    REPO_DIR = Path.cwd()
    os.chdir(REPO_DIR)
    if (REPO_DIR / '.git').exists():
        fetch_result = _run_setup('git fetch origin', check=False)
        if fetch_result.returncode == 0:
            _pull_or_continue(BRANCH)
        else:
            print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    _run_setup(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

os.chdir(REPO_DIR)
_run_setup('git status --short --branch', check=False)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('chapman_zip:', chapman_zip, '| size=', chapman_zip.stat().st_size)
print('model_dir  :', model_dir)
print('branch    :', BRANCH)

cache_status = {
    'clean_raw_cache': DRIVE_ROOT / 'ecg_data_27c_subject.npz',
    'raw_minirocket_cache': DRIVE_ROOT / 'rocket_raw_N44186_C12_L5000_K10000_S42.npz',
    'hrv36_cache': DRIVE_ROOT / 'hrv36_N44186_C12_L5000.npz',
    'fold_pca_cache_dir': DRIVE_ROOT / 'revision_feature_cache',
}
print('cache policy:')
print('  ECG_RAMBA_USE_CLEAN_CACHE =', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('  ECG_RAMBA_SAVE_CLEAN_CACHE=', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('  ECG_RAMBA_EXTRACT_DIR     =', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('cache status:')
for name, path in cache_status.items():
    if path.is_dir():
        count = len(list(path.glob('*.npz')))
        print(f'  {name}: exists=True npz_count={count} path={path}')
    else:
        print(f'  {name}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')

def run(cmd, check=True, log_path=None, tail_lines=160):
    import time

    print(f'$ {cmd}', flush=True)
    if log_path is None:
        log_dir = Path('reports/revision/logs')
        log_dir.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime('%Y%m%d_%H%M%S')
        log_path = log_dir / f'notebook_command_{stamp}.log'
    else:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open('w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = proc.wait()

    print(f'Command log: {log_path}')
    if check and return_code:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print(f'Command failed with exit code {return_code}. Last {min(tail_lines, len(lines))} log lines:')
        for line in lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')


## Install Metric Dependencies

In [ ]:
INSTALL_METRIC_DEPS = True
if INSTALL_METRIC_DEPS:
    import importlib.util
    import subprocess
    import sys

    required = {
        'numpy': 'numpy',
        'pandas': 'pandas',
        'scipy': 'scipy',
        'sklearn': 'scikit-learn',
        'threadpoolctl': 'threadpoolctl',
        'matplotlib': 'matplotlib',
        'dateutil': 'python-dateutil',
    }
    missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
    print('Python:', sys.version)
    if missing:
        print('Installing missing metric dependencies:', missing)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
    else:
        print('Metric dependencies already available; no pip install needed.')

    import matplotlib
    import numpy as np
    import scipy
    import sklearn
    print('numpy     :', np.__version__)
    print('scipy     :', scipy.__version__)
    print('sklearn   :', sklearn.__version__)
    print('matplotlib:', matplotlib.__version__)
else:
    print('Skipping metric dependency install.')


## Restore And Validate Frozen OOF Inputs

In [ ]:
stable_mirror = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
if stable_mirror.exists():
    run(
        f'python -u scripts/revision/artifact_mirror.py restore '
        f'--mirror-root "{stable_mirror}" --replace-mismatched',
        log_path='reports/revision/logs/calibration_mirror_restore.log',
    )
else:
    print('Stable mirror does not exist yet:', stable_mirror)

run(
    'python -u scripts/revision/06_freeze_oof.py '
    '--expected-records 44186 --expected-folds 5 --q 3 --check-only',
    log_path='reports/revision/logs/calibration_oof_input_check.log',
)


## Run Calibration And Bootstrap CI

In [ ]:
N_BOOT = 1000
N_BINS = 15
THRESHOLD = 0.5
OVERWRITE_METRICS = True

pred = Path('reports/revision/predictions/oof_full_predictions.npz')
freeze = Path('reports/revision/manifests/oof_freeze_manifest.json')
out = Path('reports/revision/metrics/calibration_ci_oof_full_predictions.json')

if not pred.exists() or not freeze.exists():
    raise FileNotFoundError('Notebook 02 must produce and freeze the canonical OOF artifacts first.')

cmd = (
    f'python -u scripts/revision/04_calibration_ci.py '
    f'--predictions "{pred}" --out "{out}" '
    f'--freeze-manifest "{freeze}" --require-manuscript-ready '
    f'--threshold {THRESHOLD} --n-bins {N_BINS} --n-boot {N_BOOT}'
)
if out.exists() and not OVERWRITE_METRICS:
    print('SKIP existing:', out)
else:
    run(cmd, log_path='reports/revision/logs/oof_calibration_ci.log')


## Summarize Metric Files

In [ ]:
metric_jsons = sorted(Path('reports/revision/metrics').glob('calibration_ci_*.json'))
if not metric_jsons:
    raise FileNotFoundError('No calibration_ci_*.json files found.')

for path in metric_jsons:
    payload = json.loads(path.read_text(encoding='utf-8'))
    print('\n' + str(path))
    print('dataset:', payload.get('dataset'))
    print('shape:', payload.get('shape'))
    print('metrics:', payload.get('metrics'))
    print('calibration:', payload.get('calibration'))
    print('calibration_micro:', payload.get('calibration_micro'))
    print('reliability:', payload.get('reliability'))
    print('bootstrap_ci:', payload.get('bootstrap_ci'))
    print('artifacts:', payload.get('artifacts'))


## Build Reviewer Tables

In [ ]:
import pandas as pd
from IPython.display import display

table_dir = Path('reports/revision/tables')
table_dir.mkdir(parents=True, exist_ok=True)

calibration_rows = []
bootstrap_rows = []

for path in sorted(Path('reports/revision/metrics').glob('calibration_ci_*.json')):
    payload = json.loads(path.read_text(encoding='utf-8'))
    metrics = payload.get('metrics', {})
    calibration = payload.get('calibration', {})
    calibration_micro = payload.get('calibration_micro', {})
    reliability = payload.get('reliability', {})
    shape = payload.get('shape', {})
    row = {
        'dataset': payload.get('dataset') or Path(payload.get('predictions', path.stem)).stem,
        'predictions': payload.get('predictions'),
        'protocol': payload.get('protocol'),
        'git_commit': payload.get('git_commit'),
        'seed': payload.get('seed'),
        'n_records': shape.get('y_true', [None, None])[0],
        'n_classes': shape.get('y_true', [None, None])[1],
        'threshold': payload.get('threshold'),
        'n_bins': payload.get('n_bins'),
        'n_boot': payload.get('n_boot'),
        'reliability_scope': reliability.get('scope'),
    }
    row.update(metrics)
    row.update(calibration)
    row.update(calibration_micro)
    calibration_rows.append(row)

    for metric_name, ci in payload.get('bootstrap_ci', {}).items():
        bootstrap_rows.append({
            'dataset': row['dataset'],
            'metric': metric_name,
            'mean': ci.get('mean'),
            'ci_low': ci.get('lo'),
            'ci_high': ci.get('hi'),
            'n_boot_valid': ci.get('n_boot_valid'),
            'predictions': row['predictions'],
            'git_commit': row['git_commit'],
        })

table_calibration = table_dir / 'table_calibration.csv'
table_bootstrap = table_dir / 'table_bootstrap_ci.csv'
pd.DataFrame(calibration_rows).to_csv(table_calibration, index=False)
pd.DataFrame(bootstrap_rows).to_csv(table_bootstrap, index=False)

print('Wrote:', table_calibration)
print('Wrote:', table_bootstrap)
print('\nCalibration table preview:')
display(pd.DataFrame(calibration_rows))
print('\nBootstrap CI table preview:')
display(pd.DataFrame(bootstrap_rows))


In [ ]:
!python scripts/revision/05_artifact_inventory.py


## Mirror Metrics To Stable Drive Cache

In [ ]:
source_root = Path('reports/revision')
mirror_root = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
run(
    f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{mirror_root}"',
    log_path='reports/revision/logs/calibration_mirror_publish.log',
)
